# NERISAL: export head models to ExecuTorch (.pte), Google Colab

Same path that worked in round one. Run on **Colab** (CPU is fine).
1. Run cell 1, then upload `nerisal_head_models.zip` from the Kaggle output (file panel on the left, or cell 2).
2. Run the rest. Download `nerisal_head_pte.zip` at the end and give it to the Builder.

In [ ]:
# 1. Install: pin ExecuTorch to the version on our Pi (1.6.0)
!pip install -q ultralytics executorch==1.6.0
!pip show executorch | grep Version

In [ ]:
# 2. Upload the Kaggle output zip (skip if you dragged it into the file panel)
import os, zipfile
if not os.path.exists("nerisal_head_models.zip"):
    from google.colab import files
    files.upload()
zipfile.ZipFile("nerisal_head_models.zip").extractall("models")
print(os.listdir("models"))

In [ ]:
# 3. Export both models to .pte (FP32, XNNPACK)
import shutil, glob
from ultralytics import YOLO
os.makedirs("pte", exist_ok=True)
for name, size in (("head640", 640), ("head320", 320)):
    w = f"models/{name}/best.pt"
    if not os.path.exists(w):
        print("missing", w); continue
    out = YOLO(w).export(format="executorch", imgsz=size)
    pte = glob.glob(f"{out}/*.pte") if os.path.isdir(out) else [out]
    shutil.copy(pte[0], f"pte/{name}.pte")
    print(name, "->", f"pte/{name}.pte", f"{os.path.getsize(f'pte/{name}.pte')/1e6:.1f} MB")

In [ ]:
# 4. Check each .pte runs and has the head-model output shape (1, 5, N): 4 box values + 1 class
import torch
from executorch.runtime import Runtime
for name, size, n in (("head640", 640, 8400), ("head320", 320, 2100)):
    p = f"pte/{name}.pte"
    if not os.path.exists(p): continue
    m = Runtime.get().load_program(p).load_method("forward")
    out = m.execute([torch.rand(1, 3, size, size)])[0]
    print(name, tuple(out.shape), "OK" if tuple(out.shape) == (1, 5, n) else "UNEXPECTED: tell Claude")

In [ ]:
# 5. Zip for the Builder
with zipfile.ZipFile("nerisal_head_pte.zip", "w") as z:
    for p in glob.glob("pte/*.pte"):
        z.write(p, os.path.basename(p))
print(zipfile.ZipFile("nerisal_head_pte.zip").namelist())
from google.colab import files; files.download("nerisal_head_pte.zip")